# Configuración del entorno en Colab

Ejecutar al inicio de **cada sesión** de Colab. Hace lo siguiente:

1. Verifica la GPU asignada.
2. Monta Google Drive.
3. Clona el repositorio (o lo actualiza si ya existe).
4. Instala las dependencias.
5. Copia `cct_images.tar.gz` de LILA a Drive (**solo la primera vez**; luego se salta).
6. Extrae el dataset desde Drive al disco local de Colab (`/content`), en `data/raw/`.
7. Verifica el número de imágenes y anotaciones.
8. Genera las etiquetas YOLO y las particiones (`src/data/coco_to_yolo.py`) en `data/yolo/`.

> Se entrena leyendo desde el disco local, no desde Drive: leer miles de imágenes directo desde Drive es muy lento y deja a la GPU esperando.

Se usa el **CCT completo en resolución original** (`cct_images.tar.gz`, 105 GB; pese a la extensión es un `.tar` sin comprimir). Extraído ocupa unos 105 GB, así que la máquina de Colab debe tener disco suficiente: la celda 6 lo verifica antes de empezar. La extracción tarda bastante.

**Estructura esperada en Drive** (ajustar `DRIVE_DATA_DIR` si es otra):
```
MyDrive/redes-parcial-p3/
  cct_images.tar.gz                  imágenes del CCT completo (105 GB; lo crea la celda 5)
  annotations/
    caltech_bboxes_20200316.json     bounding boxes
    caltech_camera_traps.json.zip    etiquetas por imagen, fechas, secuencias, cámaras
    CaltechCameraTrapsSplits_v0.json partición por cámara (incluye eccv_train / eccv_val)
    eccv_18_annotations.tar.gz       splits ECCV 2018 (cis / trans)
```

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / actualizar el repositorio

In [ ]:
import os

REPO_URL = 'https://github.com/Roxana-Vargas/redes-parcial-p3.git'
REPO_DIR = '/content/redes-parcial-p3'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 4. Dependencias

In [ ]:
!pip install -q -r requirements.txt

import ultralytics
ultralytics.checks()

## 5. Copiar el dataset de LILA a Drive (solo la primera vez)

Descarga `cct_images.tar.gz` directo desde el bucket público de LILA a Drive (de servidor a servidor, sin pasar por la PC). Si el archivo ya está completo en Drive, la celda no hace nada. Si se corta, volver a ejecutarla: continúa donde quedó.

Se verifica que el tamaño coincida exactamente con el del servidor (112,540,764,160 bytes).

In [ ]:
LILA_URL = 'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/cct_images.tar.gz'
EXPECTED_BYTES = 112_540_764_160

DRIVE_DATA_DIR = '/content/drive/MyDrive/redes-parcial-p3'
IMAGES_TAR = os.path.join(DRIVE_DATA_DIR, 'cct_images.tar.gz')


def tar_size():
    return os.path.getsize(IMAGES_TAR) if os.path.isfile(IMAGES_TAR) else 0


if tar_size() == EXPECTED_BYTES:
    print('cct_images.tar.gz ya está completo en Drive.')
else:
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    print(f'En Drive: {tar_size() / 1e9:.1f} GB de {EXPECTED_BYTES / 1e9:.1f} GB. Descargando de LILA...')
    !curl -L -C - --retry 10 --retry-delay 5 -o "{IMAGES_TAR}" "{LILA_URL}"
    # Forzar que Drive termine de escribir antes de verificar
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    assert tar_size() == EXPECTED_BYTES, (
        f'Tamaño incorrecto: {tar_size():,} bytes (se esperaban {EXPECTED_BYTES:,}). '
        'Volver a ejecutar esta celda para continuar la descarga.')
    print('Descarga completa y verificada.')

## 6. Extraer el dataset en `/content`

In [ ]:
import glob, shutil

RAW_DIR = os.path.join(REPO_DIR, 'data', 'raw')
DONE_FLAG = os.path.join(RAW_DIR, '.images_ok')
MIN_FREE_GB = 120  # 105 GB de imágenes + margen para labels YOLO y salidas

assert tar_size() == EXPECTED_BYTES, 'cct_images.tar.gz no está completo en Drive: ejecutar la celda 5'
os.makedirs(RAW_DIR, exist_ok=True)

# Anotaciones: archivos pequeños, se copian y se descomprimen
ann_dst = os.path.join(RAW_DIR, 'annotations')
shutil.copytree(os.path.join(DRIVE_DATA_DIR, 'annotations'), ann_dst, dirs_exist_ok=True)
for z in glob.glob(os.path.join(ann_dst, '*.zip')):
    !unzip -oq "{z}" -d "{ann_dst}"
for t in glob.glob(os.path.join(ann_dst, '*.tar.gz')):
    !tar -xzf "{t}" -C "{ann_dst}"

# Imágenes: se extraen directo desde Drive (sin copiar antes el archivo) en data/raw/cct_images/
if os.path.exists(DONE_FLAG):
    print('Imágenes ya extraídas en esta sesión.')
else:
    free_gb = shutil.disk_usage('/content').free / 1e9
    print(f'Disco libre en /content: {free_gb:.0f} GB')
    assert free_gb > MIN_FREE_GB, (
        f'Se necesitan más de {MIN_FREE_GB} GB libres. '
        'Cambiar a un entorno con más disco (L4 / A100).')
    !apt-get -qq install -y pv > /dev/null
    # LILA lo publica como .tar.gz pero es un .tar sin comprimir: se detecta por los primeros bytes
    with open(IMAGES_TAR, 'rb') as f:
        is_gzip = f.read(2) == b'\x1f\x8b'
    if is_gzip:
        !pv -f "{IMAGES_TAR}" | tar -xz -C "{RAW_DIR}"
    else:
        !pv -f "{IMAGES_TAR}" | tar -x -C "{RAW_DIR}"
    open(DONE_FLAG, 'w').close()

!du -sh {RAW_DIR}/*

## 7. Verificación rápida

In [ ]:
n_images = sum(1 for p in glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True))
json_files = glob.glob(os.path.join(RAW_DIR, '**', '*.json'), recursive=True)

print(f'Imágenes .jpg: {n_images:,}')
print('Anotaciones JSON:')
for p in sorted(json_files):
    print('  ', os.path.relpath(p, RAW_DIR))

## 8. Etiquetas YOLO y particiones

Convierte las anotaciones COCO a formato YOLO y genera las listas de cada partición y los `data_*.yaml` en `data/yolo/`.

In [ ]:
!python src/data/coco_to_yolo.py
!ls data/yolo